# Anomaly Detection at Redox-Based Bio-Cyber Interfaces

Reproduction code for *Hybrid Deep Learning Techniques for Securing Redox Based
Interfaces in Internet of Bio Nano Things*.

Sidra Zafar, Saba Iqbal, Asghar Ali Shah, Sohail Jabbar, Zia Ul Rehman

---

The corpus analysed in this study is synthetic, generated from an
electrochemical forward model rather than collected experimentally. The
generator is provided in `rbi_all.py`, so the exact corpus behind every
reported result can be reconstructed from a fixed seed.

### Modules

| File | Contents |
|---|---|
| `rbi_all.py` | Forward model, attack transformations, corpus generation, six architectures, six baselines, cross-validation protocol, ablation |
| `rbi_eda.py` | Exploratory analysis of the window-level feature representation |
| `rbi_report.py` | Tables and figures from a completed run |

### Reproducibility

Corpus generation is deterministic: `generate_corpus(n_windows=6000, seed=42)`
returns the identical corpus on any platform, so the dataset, the fidelity
analysis, and all classical and statistical baselines reproduce exactly.

The deep architectures do not reproduce bit-for-bit. GPU training through cuDNN
uses non-deterministic reduction ordering in convolution and recurrent kernels,
so independent runs under an identical seed differ by roughly one to three
percentage points in mean F1. Results are therefore reported as cross-validated
means with confidence intervals rather than single point estimates.

### Runtime on an NVIDIA T4

| Step | Time |
|---|---|
| Fidelity analysis | under a minute, no GPU required |
| Main experiment, 6,000 windows, 10 folds | 1–2 hours |
| Resampling ablation, three conditions | 9–12 hours |

Each run checkpoints after every fold and resumes if interrupted.

## Setup

Upload `rbi_all.py`, `rbi_eda.py`, and `rbi_report.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()

!ls -la *.py

Mount Drive so results survive a disconnected session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/rbi'
!mkdir -p {BASE}

## Dataset fidelity analysis

Four physical consistency checks on the generator, reported in Section 4.1.5
and Figure 4. The cyclic voltammogram is obtained by solving the Volterra
integral equation of semi-infinite linear diffusion, so Randles–Sevcik
behaviour emerges from the solver rather than being imposed on it.

Expected: Randles–Sevcik $R^2 = 1.0000$ with slope within 2.13% of theory;
Tafel transfer coefficient 0.5001 against an input of 0.5000; Cottrell
$R^2 = 1.0000$; peak separation 61.6 mV against the 59 mV reversible limit.

In [ ]:
!python rbi_all.py --mode fidelity

## Main experiment

Ten-fold stratified cross-validation across six deep architectures and six
classical and statistical baselines, on identical folds with identical
preprocessing and threshold calibration. Produces Tables 8 and 9 and
Figures 11 to 14.

The output directory below holds the run reported in the paper. Because the
deep architectures are not bit-reproducible, re-running will not return
identical numbers, so the cell is left commented. Uncomment to generate a
fresh run, writing to a new directory.

In [ ]:
# !python rbi_all.py --mode experiment --n 6000 --folds 10 \
#     --out {BASE}/results_new

## Resampling ablation

Three conditions, applied to the fit split of each training fold:

- `none` — no resampling
- `smote_raw` — SMOTE interpolated between raw sequences
- `window` — window oversampling with jitter and circular shift

The classical baselines receive feature-space SMOTE under both resampled
conditions, so that only the sequence-model treatment varies. `window` is the
condition used for the primary results. Produces Table 7 and Figure 10.

In [ ]:
# !python rbi_all.py --mode ablation --n 6000 --folds 10 \
#     --out {BASE}/ablation_new

## Exploratory analysis

Descriptive statistics and distribution plots for the 20-dimensional
window-level feature representation. Produces Table 5 and Figures 5 to 9.

In [ ]:
!python rbi_eda.py --n 6000 --out {BASE}/paper_assets

## Model size and inference cost

Parameter counts and per-window inference latency for each architecture,
reported as Table 10. Latency is measured in batches of 256 and is a relative
measure of computational demand rather than a deployment figure.

In [ ]:
import numpy as np, time, tensorflow as tf
from rbi_all import build_model, DL_MODELS

print(f"{'Model':<10} {'Params':>10} {'ms/window':>12}")
for name in DL_MODELS:
    tf.keras.backend.clear_session()
    m = build_model(name, 256)
    x = np.random.randn(256, 256, 1).astype('float32')
    m.predict(x[:32], verbose=0)          # warm up
    t0 = time.perf_counter()
    m.predict(x, verbose=0)
    ms = (time.perf_counter() - t0) * 1000 / 256
    print(f"{name:<10} {m.count_params():>10,} {ms:>12.3f}")

## Tables and figures

Everything below is derived from `per_fold_metrics.csv`, so the tables and the
figures cannot disagree. Produces Tables 8 and 9 as LaTeX, together with the
F1 comparison, the accuracy against ROC-AUC scatter, and the confusion
matrices.

In [ ]:
# ============================================================
# Regenerate results figures and tables from per_fold_metrics.csv
# ============================================================
import json, os
import numpy as np, pandas as pd
from scipy import stats as sps
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

RESULTS = '/content/drive/MyDrive/rbi/results_v2'
OUT     = '/content/drive/MyDrive/rbi/paper_final'
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "DejaVu Serif"],
                     "font.size": 10, "axes.linewidth": 0.8})

ORDER = ["1D-CNN","CNN","GRU","LSTM","CNN+GRU","CNN+LSTM","RandomForest",
         "IsolationForest","OneClassSVM","CUSUM","EWMA","NaiveThreshold"]
DEEP  = ORDER[:6]
LABEL = {"RandomForest":"Random Forest","IsolationForest":"Isolation Forest",
         "OneClassSVM":"One-Class SVM","NaiveThreshold":"Naive threshold"}
METRICS = ["accuracy","precision","recall","f1","fpr","roc_auc","mse","mae"]
HEADERS = ["Accuracy","Precision","Recall","F1","FPR","ROC-AUC","MSE","MAE"]
LOWER   = {"fpr","mse","mae"}
C_DEEP, C_BASE = "#2b5d8a", "#a3a3a3"
nm = lambda m: LABEL.get(m, m)

# ---------- load ----------
d = pd.read_csv(f"{RESULTS}/per_fold_metrics.csv")
present = [m for m in ORDER if m in set(d.model)]
n_folds = d.fold.nunique()
summary = {m: {k: (d[d.model==m][k].mean(), d[d.model==m][k].std(ddof=1))
               for k in METRICS} for m in present}
cm = {m: [[int(d[d.model==m].tn.sum()), int(d[d.model==m].fp.sum())],
          [int(d[d.model==m].fn.sum()), int(d[d.model==m].tp.sum())]]
      for m in present}
pf   = d.pivot(index="fold", columns="model", values="f1")
best = max(present, key=lambda m: summary[m]["f1"][0])

# ---------- significance ----------
def holm(p):
    p = np.asarray(p, float); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (len(p)-r)*p[i]); adj[i] = min(run, 1.0)
    return adj

rows = []
for m in [x for x in pf.columns if x != best]:
    a, b = pf[best].values, pf[m].values
    diff = a - b
    try: w = sps.wilcoxon(a, b).pvalue
    except ValueError: w = 1.0
    pooled = np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2) or 1e-12
    rows.append(dict(model=m, mean_diff=diff.mean(),
                     cohens_d=diff.mean()/pooled,
                     t_p=sps.ttest_rel(a,b).pvalue, w_p=w))
sig = pd.DataFrame(rows)
sig["t_p_holm"] = holm(sig.t_p); sig["w_p_holm"] = holm(sig.w_p)
sig["significant"] = sig.t_p_holm < 0.05
sig = sig.sort_values("mean_diff").reset_index(drop=True)

# ---------- Figure: F1 with 95% CI ----------
half = sps.t.ppf(0.975, n_folds-1)/np.sqrt(n_folds)
mean = np.array([summary[m]["f1"][0] for m in present])
err  = np.array([summary[m]["f1"][1] for m in present])*half
fig, ax = plt.subplots(figsize=(7.2,4.2))
ax.bar(np.arange(len(present)), mean, yerr=err, capsize=3,
       color=[C_DEEP if m in DEEP else C_BASE for m in present],
       edgecolor="black", linewidth=0.6, width=0.65)
ax.set_xticks(np.arange(len(present)))
ax.set_xticklabels([nm(m) for m in present], rotation=40, ha="right", fontsize=8.5)
ax.set_ylabel("F1 score"); ax.set_ylim(0,1)
ax.set_title("Detection performance across architectures and baselines\n"
             "(dark bars: deep architectures; grey: classical/statistical baselines)",
             fontsize=9.5)
ax.yaxis.grid(True, linewidth=0.4, alpha=0.5); ax.set_axisbelow(True)
for s in ("top","right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_f1_comparison.png", dpi=600,
                                bbox_inches="tight"); plt.close()

# ---------- Figure: accuracy vs ROC-AUC ----------
POS = {"CNN+GRU":(0.560,0.968,"right"), "CNN+LSTM":(0.560,0.930,"right"),
       "RandomForest":(0.560,0.892,"right"), "1D-CNN":(0.560,0.854,"right"),
       "CNN":(0.876,0.845,"left"), "OneClassSVM":(0.645,0.812,"right"),
       "GRU":(0.848,0.760,"left"), "IsolationForest":(0.700,0.723,"left"),
       "CUSUM":(0.700,0.686,"left"), "LSTM":(0.700,0.649,"left"),
       "EWMA":(0.590,0.600,"left"), "NaiveThreshold":(0.480,0.556,"left")}
fig, ax = plt.subplots(figsize=(6.4,5.2))
for m in present:
    px, py = summary[m]["accuracy"][0], summary[m]["roc_auc"][0]
    lx, ly, ha = POS.get(m, (px+0.02, py+0.01, "left"))
    ax.scatter(px, py, s=58, color=C_DEEP if m in DEEP else C_BASE,
               marker="o" if m in DEEP else "s", edgecolor="black",
               linewidth=0.7, zorder=4, alpha=0.92)
    ax.annotate(nm(m), xy=(px,py), xytext=(lx,ly), textcoords="data",
                fontsize=8, ha=ha, va="center", zorder=5,
                arrowprops=dict(arrowstyle="-", linewidth=0.55,
                                color="#666666", shrinkA=2, shrinkB=6))
ax.set_xlabel("Accuracy"); ax.set_ylabel("ROC-AUC")
ax.set_title("Accuracy versus ROC-AUC by model", fontsize=10.5)
ax.set_xlim(0.28,0.96); ax.set_ylim(0.53,0.99)
ax.grid(True, linewidth=0.4, alpha=0.45); ax.set_axisbelow(True)
for s in ("top","right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_roc_accuracy.png", dpi=600,
                                bbox_inches="tight"); plt.close()

# ---------- Figures: confusion matrices ----------
cmap = LinearSegmentedColormap.from_list("b", ["#ffffff","#2b5d8a"])
def draw(ax, M, title, ylab=True, xlab=True):
    M = np.array(M, float); rn = M/M.sum(1, keepdims=True)
    ax.imshow(rn, cmap=cmap, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(M[i,j]):,}\n({rn[i,j]*100:.1f}%)",
                    ha="center", va="center", fontsize=8,
                    color="white" if rn[i,j]>0.55 else "#1a1a1a")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Benign","Attack"], fontsize=8)
    ax.set_yticklabels(["Benign","Attack"], fontsize=8, rotation=90, va="center")
    if xlab: ax.set_xlabel("Predicted", fontsize=8.5)
    if ylab: ax.set_ylabel("Actual", fontsize=8.5)
    ax.set_title(title, fontsize=9.5, pad=6); ax.tick_params(length=0)

fig, ax = plt.subplots(figsize=(3.3,3.1)); draw(ax, cm[best], nm(best))
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_confusion_best.png", dpi=600,
                                bbox_inches="tight"); plt.close()

deep = [m for m in DEEP if m in cm]
fig, axes = plt.subplots(2,3, figsize=(7.4,5.0))
for k,(a_,m) in enumerate(zip(axes.ravel(), deep)):
    draw(a_, cm[m], nm(m), ylab=(k%3==0), xlab=(k>=3))
plt.tight_layout(pad=1.1); plt.savefig(f"{OUT}/Fig_confusion_all_deep.png",
                                       dpi=600, bbox_inches="tight"); plt.close()

# ---------- Table 8 ----------
bestcol = {k: (min if k in LOWER else max)(
    {m: summary[m][k][0] for m in present}.items(), key=lambda x: x[1])[0]
    for k in METRICS}
L = [r"\begin{table*}[htbp]", r"\centering",
     r"\caption{Classification performance over "+str(n_folds)+
     r"-fold stratified cross-validation. Values are mean $\pm$ standard deviation "
     r"across folds. Deep architectures are listed above the horizontal rule; "
     r"classical and statistical baselines below. Best value in each column is shown "
     r"in bold. FPR, MSE, and MAE are error measures for which lower values are better.}",
     r"\label{tab:results}", r"\footnotesize", r"\setlength{\tabcolsep}{4pt}",
     r"\begin{tabular}{l"+"c"*len(METRICS)+"}", r"\toprule",
     r"\textbf{Model} & "+" & ".join(r"\textbf{"+h+"}" for h in HEADERS)+r" \\",
     r"\midrule"]
for m in present:
    if m == "RandomForest": L.append(r"\midrule")
    cells = []
    for k in METRICS:
        mu, sd = summary[m][k]
        c = f"{mu:.3f} $\\pm$ {sd:.3f}"
        if bestcol[k] == m: c = r"\textbf{"+c+"}"
        cells.append(c)
    L.append(nm(m)+" & "+" & ".join(cells)+r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
open(f"{OUT}/table8_results.tex","w").write("\n".join(L))

# ---------- Table 9 ----------
fmt = lambda p: "$<$0.0001" if p < 0.0001 else f"{p:.4f}"
L = [r"\begin{table}[htbp]", r"\centering",
     r"\caption{Paired comparisons against "+nm(best)+r" across "+str(n_folds)+
     r" cross-validation folds, F1 score. Positive $\Delta$F1 indicates that "+
     nm(best)+r" outperforms the comparator. Holm correction is applied across the "+
     str(len(sig))+r" comparisons.}",
     r"\label{tab:significance}", r"\footnotesize", r"\setlength{\tabcolsep}{4pt}",
     r"\begin{tabular}{lcccc}", r"\toprule",
     r"\textbf{Comparator} & $\Delta$\textbf{F1} & \textbf{Cohen's} $d$ & "
     r"\textbf{$t$-test} $p_{\text{Holm}}$ & \textbf{Wilcoxon} $p_{\text{Holm}}$ \\",
     r"\midrule"]
for _, r in sig.iterrows():
    L.append(f"{nm(r.model)} & {r.mean_diff:.4f} & {r.cohens_d:.2f} & "
             f"{fmt(r.t_p_holm)} & {fmt(r.w_p_holm)} "+r"\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(f"{OUT}/table9_significance.tex","w").write("\n".join(L))

pd.DataFrame([{"model":m,"metric":k,"mean":round(summary[m][k][0],4),
               "sd":round(summary[m][k][1],4)}
              for m in present for k in METRICS]).to_csv(
    f"{OUT}/summary_mean_sd.csv", index=False)
json.dump(cm, open(f"{OUT}/confusion_matrices.json","w"), indent=2)

# ---------- report ----------
print(f"folds {n_folds} | models {len(present)} | best by F1: {best}\n")
print("ranking by mean F1")
for m in sorted(present, key=lambda x: -summary[x]["f1"][0]):
    mu, sd = summary[m]["f1"]; print(f"  {nm(m):<18s}{mu:.4f} +/- {sd:.4f}")
tn, fp = cm[best][0]; fn, tp = cm[best][1]
print(f"\n{nm(best)}:  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"  accuracy {(tn+tp)/(tn+fp+fn+tp):.4f}   missed attacks {fn/(fn+tp)*100:.1f}%")
ns = (~sig.significant).sum()
print(f"\nsignificance: {len(sig)} comparisons, {ns} not significant after Holm")
if ns: print("  ", ", ".join(sig[~sig.significant].model))
if np.allclose(sig.w_p_holm, sig.w_p_holm.min()):
    print(f"  all Holm-corrected Wilcoxon values at {sig.w_p_holm.min():.4f}, "
          f"the floor for {n_folds} paired folds")

print("\n"+"="*62+"\nTABLE 8\n"+"="*62)
print(open(f"{OUT}/table8_results.tex").read())
print("="*62+"\nTABLE 9\n"+"="*62)
print(open(f"{OUT}/table9_significance.tex").read())
print(f"\nwritten to {OUT}/")

## Archive the outputs

In [ ]:
!zip -r {BASE}/paper_assets.zip {BASE}/paper_assets

---

### Citation

```bibtex
@article{zafar2026redox,
  title   = {Hybrid Deep Learning Techniques for Securing Redox Based
             Interfaces in Internet of Bio Nano Things},
  author  = {Zafar, Sidra and Iqbal, Saba and Shah, Asghar Ali and
             Jabbar, Sohail and Rehman, Zia Ul},
  journal = {Engineering Reports},
  year    = {2026}
}
```